# Goal

Почему не сработали `17e_study_14.1` и `17e_study_15.1`? Временно пробуем логику с game_modifs, которая была `17e_ppo_tr_atarai_mp_12`. Неужели проблема в RAM?

# set_hyperparameters

In [2]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 6
    generation_ind = 0
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    ####
    
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    HP.env.idle_penalty = 0
    HP.env.life_lost_penalty = 0

    HP.agent.parent = None
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = None
    HP.ppo.rollout_game_modifs = [
        ['reset_last_life', 'reset_full_igloo', 'reset_bailey_right_at_the_igloo_door', 'reset_temperature_10'],
        ['reset_last_life', 'reset_full_igloo', 'reset_bailey_very_near_igloo_door', 'reset_temperature_10'],  
        ['reset_last_life', 'reset_full_igloo', 'reset_bailey_near_center', 'reset_temperature_10'],
        ['reset_last_life', 'reset_one_remaining_igloo', 'reset_bailey_near_center', 'reset_temperature_10'],
        ['reset_last_life', 'reset_three_remaining_igloo', 'reset_bailey_near_center', 'reset_temperature_20'],
        ['reset_last_life', 'reset_half_igloo', 'reset_bailey_near_center'],
        ['reset_last_life', 'reset_half_igloo'],
        ['reset_last_life'],
    ]
    HP.ppo.tau = 'linear(0.5, 0.34)' # temperature to inject randomness during actions selection (Gumbel Max)
    
    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 8
    HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # e target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

In [5]:
from unittest.mock import Mock
HP = Mock()
set_hyperparameters(HP, None, None)
HP.ppo.tau, HP.ppo.learn_rate, HP.ppo.ent_coef

('linear(0.5, 0.43333333333333335)',
 'linear(0.00025, 0.00021250000000000002)',
 'linear(0.05, 0.0425)')

# Results
<TBD>

С возвращённой логикой game_modifs агент сумел сделать прорыв. После RAM vs game_modifs обнаружилась явная ошибка: исходный RAM многократно перезатирался, что приводило к тому, что а) мы использовалис RAM со всеми предыдущими патчами в непонятном порядке, б) число стартовых состояний уменьшалось.

Фик тривиальный:
```
--- a/17_rl/17e_ppo_tr_atari_mp_13.ipynb
+++ b/17_rl/17e_ppo_tr_atari_mp_13.ipynb
@@ -3882,6 +3882,7 @@
     "        ram_name = RNG.choice(list(self.rams.keys()))\n",
     "        ram = self.rams[ram_name]\n",
     "        assert isinstance(ram, np.ndarray)\n",
+    "        ram = ram.copy() # mandatory to not spoil original RAM!\n",
     "        patch_desc = ''\n",
     "\n",
     "        if self.patches is not None and self.patches:\n",
```

<img src="./img/reward.png">

**Вывод**. После фикса можно вернуться к повтору `17e_study_14.1`